# 00_a Prep Ukraine data

Builds the Ukrainian layers the rest of the pipeline reads, from public sources only (no local
database). **Run this before `00_extract_base_data_eu.ipynb`**, which appends these hromadas
to the LAU 2024 subset (2a), takes their DEGURBA (2b) and uses their population for GDP per
head (3), and reads `ukr_admin1.shp` for the UA oblasts (1a).

**Writes**
- `data/ukraine/ukr_admin4_population.gpkg`, layer `settlements` -- 29,706 settlements with
  population (1 Jan 2022) and `degurba_l1_ghsl`.
- `data/ukraine/ukr_admin3_population.gpkg`, layer `hromadas` -- 1,769 hromadas (Ukraine's LAU
  equivalent) with population, `degurba_l1_ghsl` (area rule, used downstream) and
  `degurba_l1_pop` (official population rule, kept as a robustness check).

**Sources** (downloaded by section 1 if missing)

| File (under `data/ukraine/`) | Source | Licence |
|---|---|---|
| `admin_boundaries/ukr_admin{0,1,3,4}.*`, `ukr_admincapitals.*` | OCHA / HDX, *Ukraine - Subnational Administrative Boundaries* (COD-AB), <https://data.humdata.org/dataset/cod-ab-ukr>, resource `ukr_admin_boundaries.shp.zip` | CC BY-IGO |
| `225-chiselnist-naiavnogo-naselennia-...-gromadakh.xlsx` | State Statistics Service of Ukraine, population estimate by regions, raions, territorial hromadas and settlements, 1 Jan 2022, <https://data.gov.ua/dataset/46ca2741-b4e8-4939-98ec-83199de5e502> | CC BY 4.0 |
| `14-valovii-regionalnii-produkt-u-faktichnikh-tsinakh-mln-grn.xlsx` (read by `00_extract_base_data_eu` section 3) | State Statistics Service of Ukraine, gross regional product at actual prices, million UAH, <https://data.gov.ua/dataset/b5b74954-eb45-4e0b-bdf0-273a881b636e> | CC BY 4.0 |
| `degurba/UKR_DUG_2024_GRID_L1_R2025A_v1.tif` | WorldPop R2025A v1, Degree of Urbanisation (GHS-DUG) grid L1, 2024, <https://hub.worldpop.org/geodata/summary?id=125970>, DOI 10.5258/SOTON/WP00879 | CC BY 4.0 |

Derived from `input_data_extension_ukraine.ipynb` without its local-database sections (the UA NUTS rows
now live in the `00_extract_base_data_eu` subset files) and without its GDP section (moved to
`00_extract_base_data_eu` section 3).

## Setup

Project-root working directory and `cfg`. Importing `traintran` also points PROJ at this
Python env's own `proj.db` (a machine-wide PROJ path set by another program breaks every CRS).

In [1]:
import sys, os

# Ensure CWD is the project root (one level up from notebooks/)
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

import traintran                     # sets PROJ_DATA to this env's proj.db, before geopandas
from traintran.config import default_config
cfg = default_config()

import pandas as pd
import geopandas as gpd

Working directory: e:\notes\api_zssk\project


## 1. Download raw inputs

Each file is fetched only if it is missing (prints `[exists]` otherwise). The HDX boundaries
are looked up through HDX's public CKAN API (`package_show`), so a re-issued resource id
does not break the link; the zip's shapefiles are extracted into `admin_boundaries/` and the
zip is deleted. data.gov.ua may refuse automated downloads -- if it does, the cell stops and
prints the direct link to download by hand.

In [2]:
import shutil
import zipfile
from pathlib import Path

import requests

HEADERS = {"User-Agent": "academic-research/1.0 (KSC rail study)"}
UA_DIR = Path("data/ukraine")
ADMIN_DIR = UA_DIR / "admin_boundaries"

def download(url, dest, force=False):
    """Stream-download `url` to `dest`. Skip if cached unless force=True."""
    dest = Path(dest)
    if dest.exists() and not force and dest.stat().st_size > 0:
        print(f"  [exists]    {dest}")
        return dest
    print(f"  [fetch]     {url}")
    with requests.get(url, headers=HEADERS, stream=True, timeout=300) as r:
        if r.status_code == 403:
            raise PermissionError(f"{r.url} refused the download (HTTP 403). Download it in a "
                                  f"browser and save it as {dest}, then re-run this cell.")
        r.raise_for_status()
        dest.parent.mkdir(parents=True, exist_ok=True)
        with open(dest, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)
    print(f"  [saved]     {dest} ({dest.stat().st_size/1e6:.1f} MB)")
    return dest

# --- Direct links
UA_FILES = {
    UA_DIR / "225-chiselnist-naiavnogo-naselennia-za-otsinkoiu-po-regionakh-raionakh-teritorialnikh-gromadakh.xlsx":
        "https://data.gov.ua/dataset/cae08549-20f4-403d-9ce0-dd306ed9095c/resource/866abfd1-a09a-45b9-b98f-a64dec94bce6/download/225-chiselnist-naiavnogo-naselennia-za-otsinkoiu-po-regionakh-raionakh-teritorialnikh-gromadakh.xlsx",
    UA_DIR / "14-valovii-regionalnii-produkt-u-faktichnikh-tsinakh-mln-grn.xlsx":
        "https://data.gov.ua/dataset/6e24b7d1-218f-4899-b187-19de3ad1ded6/resource/85b0bf89-1021-4e07-bafc-f7b7c5125555/download/14-valovii-regionalnii-produkt-u-faktichnikh-tsinakh-mln-grn.xlsx",
    UA_DIR / "degurba" / "UKR_DUG_2024_GRID_L1_R2025A_v1.tif":
        "https://data.worldpop.org/GIS/DUG/Global_2015_2030/R2025A/v1/2024/UKR/UKR_DUG_2024_GRID_L1_R2025A_v1.tif",
}
for dest, url in UA_FILES.items():
    download(url, dest)

# --- HDX COD-AB boundaries: shapefile parts of the layers this project reads.
ADMIN_LAYERS = ["ukr_admin0", "ukr_admin1", "ukr_admin3", "ukr_admin4", "ukr_admincapitals"]
SHP_PARTS = [".shp", ".shx", ".dbf", ".prj", ".cpg"]
_needed = [f"{l}{e}" for l in ADMIN_LAYERS for e in SHP_PARTS]
_missing = [f for f in _needed if not (ADMIN_DIR / f).is_file() or (ADMIN_DIR / f).stat().st_size == 0]

if not _missing:
    print(f"  [exists]    {ADMIN_DIR}/ ({', '.join(ADMIN_LAYERS)})")
else:
    HDX_DATASET = "cod-ab-ukr"
    HDX_RESOURCE = "ukr_admin_boundaries.shp.zip"
    r = requests.get("https://data.humdata.org/api/3/action/package_show",
                     params={"id": HDX_DATASET}, headers=HEADERS, timeout=60)
    r.raise_for_status()
    _res = [x for x in r.json()["result"]["resources"] if x["name"] == HDX_RESOURCE]
    if not _res:
        raise FileNotFoundError(f"{HDX_RESOURCE} not listed on https://data.humdata.org/dataset/{HDX_DATASET}")
    print(f"  HDX {HDX_RESOURCE}, last modified {_res[0].get('last_modified')}")
    hdx_zip = download(_res[0]["url"], UA_DIR / HDX_RESOURCE)
    ADMIN_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(hdx_zip) as z:
        by_name = {Path(m).name: m for m in z.namelist() if not m.endswith("/")}
        not_in_zip = [f for f in _needed if f not in by_name]
        if not_in_zip:
            raise FileNotFoundError(f"Not in {hdx_zip.name}: {not_in_zip}")
        for f in _needed:
            with z.open(by_name[f]) as s, open(ADMIN_DIR / f, "wb") as t:
                shutil.copyfileobj(s, t)
        print(f"  [extracted] {len(_needed)} files -> {ADMIN_DIR}")
    hdx_zip.unlink()
    print(f"  [deleted]   {hdx_zip}")

  [exists]    data\ukraine\225-chiselnist-naiavnogo-naselennia-za-otsinkoiu-po-regionakh-raionakh-teritorialnikh-gromadakh.xlsx
  [exists]    data\ukraine\14-valovii-regionalnii-produkt-u-faktichnikh-tsinakh-mln-grn.xlsx
  [exists]    data\ukraine\degurba\UKR_DUG_2024_GRID_L1_R2025A_v1.tif
  [exists]    data\ukraine\admin_boundaries/ (ukr_admin0, ukr_admin1, ukr_admin3, ukr_admin4, ukr_admincapitals)


## 2. Ukraine admin boundaries -- `ukr_admin0.shp` (national) and `ukr_admin1.shp` (oblast level)

Loaded as GeoDataFrames to check their CRS (expected EPSG:4326, WGS84) and columns.
`ukr_admin1` is what `00_extract_base_data_eu` appends to the NUTS subset as UA level 2.

In [3]:
import geopandas as gpd

ukr_admin0 = gpd.read_file('data/ukraine/admin_boundaries/ukr_admin0.shp')
ukr_admin1 = gpd.read_file('data/ukraine/admin_boundaries/ukr_admin1.shp')

print(f'ukr_admin0: {len(ukr_admin0)} rows, CRS = {ukr_admin0.crs}')
print(f'ukr_admin1: {len(ukr_admin1)} rows, CRS = {ukr_admin1.crs}')
print()
print('ukr_admin0 columns:', list(ukr_admin0.columns))
print('ukr_admin1 columns:', list(ukr_admin1.columns))

ukr_admin0: 1 rows, CRS = EPSG:4326
ukr_admin1: 27 rows, CRS = EPSG:4326

ukr_admin0 columns: ['iso2', 'iso3', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'version', 'area_sqkm', 'lang', 'lang1', 'lang2', 'lang3', 'center_lat', 'center_lon', 'geometry']
ukr_admin1 columns: ['adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang', 'lang1', 'lang2', 'lang3', 'center_lat', 'center_lon', 'geometry']


In [4]:
ukr_admin0.head()

,iso2,iso3,adm0_name,adm0_name1,adm0_name2,adm0_name3,adm0_pcode,valid_on,valid_to,version,area_sqkm,lang,lang1,lang2,lang3,center_lat,center_lon,geometry
0,UA,UKR,Ukraine,Україна,None,None,UA,2025-09-01,NaT,v05,600217.274418,en,uk,None,None,NaN,NaN,"MULTIPOLYGON (((31.51264 46.57787, 31.51268 46..."


In [5]:
ukr_admin1.head()

,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,adm0_name,adm0_name1,adm0_name2,adm0_name3,adm0_pcode,...,valid_to,area_sqkm,version,lang,lang1,lang2,lang3,center_lat,center_lon,geometry
0,Autonomous Republic of Crimea,Автономна Республіка Крим,None,None,UA01,Ukraine,Україна,None,None,UA,...,NaT,25558.451568,v05,en,uk,None,None,NaN,NaN,"MULTIPOLYGON (((33.70541 46.17138, 33.70322 46..."
1,Vinnytska,Вінницька,None,None,UA05,Ukraine,Україна,None,None,UA,...,NaT,26490.867508,v05,en,uk,None,None,NaN,NaN,"POLYGON ((28.87952 49.88847, 28.88087 49.88828..."
2,Volynska,Волинська,None,None,UA07,Ukraine,Україна,None,None,UA,...,NaT,20145.249915,v05,en,uk,None,None,NaN,NaN,"POLYGON ((25.22584 51.96091, 25.22584 51.96091..."
3,Dnipropetrovska,Дніпропетровська,None,None,UA12,Ukraine,Україна,None,None,UA,...,NaT,31910.688890,v05,en,uk,None,None,NaN,NaN,"POLYGON ((34.94829 49.19156, 34.94836 49.19153..."
4,Donetska,Донецька,None,None,UA14,Ukraine,Україна,None,None,UA,...,NaT,26499.602913,v05,en,uk,None,None,NaN,NaN,"POLYGON ((37.87218 49.2376, 37.87269 49.2368, ..."


## 3. Settlement population — join onto `ukr_admin4`

`ukr_admin4.shp` is the **individual settlement** level (29,706 rows: villages/settlements/
cities) -- one level finer than the hromada (`ukr_admin3`, the actual EU-LAU equivalent).
Population comes from the State Statistics Service table
`225-chiselnist-...-gromadakh.xlsx` (population by estimate, by regions/raions/territorial
hromadas *and settlements*, as of 1 Jan 2022).

The xlsx's `code` column is a KATOTTG code whose prefix is byte-identical to the shapefile's
COD pcodes at every level (`code[:12] == adm4_pcode` for settlements) -- so this is a plain
prefix join across all sheets concatenated together, no oblast-name matching or per-oblast
batching needed. Each sheet has the same layout: row 1 is the real header
(`attributes, code, data`), row 2 is a duplicate Ukrainian-language header row (skipped),
then a 4-level hierarchy per row -- oblast total -> raion total -> hromada total ->
**settlement (leaf)** -- with two extra non-administrative subtotal rows ("Miske
naselennia"/"Silske naselennia", urban/rural population) interleaved and carrying no code.
Total/non-leaf rows *do* carry a code, so isolating settlement rows needs one more filter:
their name doesn't contain "usiogo" (Ukrainian for "total" -- every non-leaf row's name ends
in that word; every leaf settlement row is a plain place name).

In [6]:
POP_XLSX = 'data/ukraine/225-chiselnist-naiavnogo-naselennia-za-otsinkoiu-po-regionakh-raionakh-teritorialnikh-gromadakh.xlsx'

pop_xl = pd.ExcelFile(POP_XLSX)
region_sheets = [s for s in pop_xl.sheet_names if s not in ('\u041e\u043f\u0438\u0441', '\u0423\u043a\u0440\u0430\u0457\u043d\u0430')]
print(f'{len(region_sheets)} region sheets (24 oblasts + Kyiv city, expected 25)')

pop_raw = pd.concat(
    [pop_xl.parse(s, header=0, skiprows=[1], usecols=[0, 1, 2]).assign(sheet=s)
     for s in region_sheets],
    ignore_index=True,
)
print(f'{len(pop_raw):,} total rows across all sheets')
print('columns:', list(pop_raw.columns))
pop_raw.head()

25 region sheets (24 oblasts + Kyiv city, expected 25)
33,523 total rows across all sheets
columns: ['attributes', 'code', 'data', 'sheet']


,attributes,code,data,sheet
0,"Вінницька область, усього",UA05000000000010236,1509515.0,Вінницька область
1,Міське населення,NaN,789588.0,Вінницька область
2,Сільське населення,NaN,719927.0,Вінницька область
3,"Вінницький район, усього",UA05020000000026686,647966.0,Вінницька область
4,Міське населення,NaN,463205.0,Вінницька область


In [7]:
# Isolate settlement leaf rows: has a code, and its name isn't a "total" row.
settlements = pop_raw[
    pop_raw['code'].notna()
    & ~pop_raw['attributes'].str.contains('\u0443\u0441\u044c\u043e\u0433\u043e', na=False)
].copy()
settlements['adm4_pcode'] = settlements['code'].str[:12]
settlements['data'] = pd.to_numeric(settlements['data'], errors='coerce')

print(f'{len(settlements):,} settlement rows isolated (expect ~28,658)')
print(f"duplicate adm4_pcode within settlements: {settlements['adm4_pcode'].duplicated().sum()}")
settlements[['attributes', 'code', 'adm4_pcode', 'data', 'sheet']].head()

28,658 settlement rows isolated (expect ~28,658)
duplicate adm4_pcode within settlements: 0


,attributes,code,adm4_pcode,data,sheet
9,с. Агрономічне,UA05020010010058853,UA0502001001,4215.0,Вінницька область
10,с. Бохоники,UA05020010020055396,UA0502001002,1838.0,Вінницька область
11,с. Горбанівка,UA05020010030044910,UA0502001003,602.0,Вінницька область
12,с. Ільківка,UA05020010040097226,UA0502001004,498.0,Вінницька область
13,с. Медвеже Вушко,UA05020010050096830,UA0502001005,1908.0,Вінницька область


In [8]:
ukr_admin4 = gpd.read_file('data/ukraine/admin_boundaries/ukr_admin4.shp')
print(f'{len(ukr_admin4):,} settlement geometries loaded (expect 29,706)')

ukr_admin4 = ukr_admin4.merge(
    settlements[['adm4_pcode', 'data']].rename(columns={'data': 'population'}),
    on='adm4_pcode', how='left',
)
print(f'{len(ukr_admin4):,} rows after merge (row count must be unchanged by a left-merge on a unique key)')

29,706 settlement geometries loaded (expect 29,706)
29,706 rows after merge (row count must be unchanged by a left-merge on a unique key)


### Match-rate check

Some `ukr_admin4` settlements won't get a population value -- the biggest expected chunk is
every settlement in occupied Crimea/Sevastopol, since the xlsx excludes that territory
entirely (no Crimea/Sevastopol sheet exists at all, matching the same occupation-exclusion
already seen in the GDP file). That's expected, not a mapping error -- flagged explicitly
below rather than left to look like an unexplained gap.

In [9]:
matched = ukr_admin4['population'].notna().sum()
unmatched = ukr_admin4[ukr_admin4['population'].isna()]
print(f'{matched:,} / {len(ukr_admin4):,} settlements matched a population value')
print(f'{len(unmatched):,} unmatched -- breakdown by oblast (adm1_name):')
print(unmatched['adm1_name'].value_counts())

# Reverse check: settlements xlsx has that never matched an admin4 pcode (expect ~5, all
# 2022-vintage codes no longer present in the current 2025-09-01 admin4 snapshot).
orphan_xlsx = settlements[~settlements['adm4_pcode'].isin(ukr_admin4['adm4_pcode'])]
print(f'\n{len(orphan_xlsx)} xlsx settlement codes have no matching admin4 pcode (expect ~5):')
orphan_xlsx[['attributes', 'code', 'sheet']]

28,653 / 29,706 settlements matched a population value
1,053 unmatched -- breakdown by oblast (adm1_name):
adm1_name
Autonomous Republic of Crimea    1050
Kyivska                             2
Sevastopol                          1
Name: count, dtype: int64

5 xlsx settlement codes have no matching admin4 pcode (expect ~5):


,attributes,code,sheet
933,с. Бабина Долина,UA05060090020070449,Вінницька область
8196,с. Затишне,UA21020030110024517,Закарпатська область
12862,с. Веселівка,UA35040090030021066,Кіровоградська область
12871,с. П’ятихатки,UA35040090120085172,Кіровоградська область
23645,с-ще Лучки,UA59060110750096198,Сумська область


In [10]:
# Spot-check a known row from the exploration: с. Банилів,
# xlsx code UA73020010010031879 (adm4_pcode = code[:12]), expected population 3641.
known_pcode = 'UA73020010010031879'[:12]
check = ukr_admin4.loc[ukr_admin4['adm4_pcode'] == known_pcode, ['adm4_pcode', 'adm4_name', 'adm4_name1', 'population']]
print(check)
assert (check['population'] == 3641).all(), 'spot-check mismatch'
print('spot-check OK')

         adm4_pcode adm4_name adm4_name1  population
27778  UA7302001001   Banyliv    Банилів      3641.0
spot-check OK


### Persist

In [11]:
OUT_GPKG = 'data/ukraine/ukr_admin4_population.gpkg'
ukr_admin4.to_file(OUT_GPKG, layer='settlements', driver='GPKG')
print(f'Wrote {OUT_GPKG}, layer=settlements, {len(ukr_admin4):,} rows')

Wrote data/ukraine/ukr_admin4_population.gpkg, layer=settlements, 29,706 rows


## 3b. Settlement DEGURBA (max) from the GHS-DUG raster

`data/ukraine/degurba/UKR_DUG_2024_GRID_L1_R2025A_v1.tif` (GHS-DUG grid, Level 1,
WorldPop R2025A v1) is a **1km, World Mollweide (`ESRI:54009`)** raster, values `{1, 2, 3}`
(nodata `0`) meaning **1=rural, 2=urban cluster, 3=urban centre** -- note this is the
*opposite* polarity from the EU vector `DGURBA` column used elsewhere in this project
(`traintran/enrichment.py::_load_dgurba`, where 1=City/dense ... 3=Rural), so the new column
here is named distinctly (`degurba_l1_ghsl`) to avoid the two scales being confused later.

Rule: where a settlement polygon overlaps multiple grid cells with different classes, assign
the **maximum** class value. Since many settlement polygons are smaller than one 1km pixel,
`all_touched=True` is used so a tiny polygon still picks up any pixel it partially overlaps
(the default `all_touched=False` only counts a pixel whose *center* falls inside the polygon,
which would wrongly return "no overlap" for many small villages).

In [12]:
import rasterio
from rasterio.mask import mask as rio_mask
import numpy as np

DEGURBA_TIF = 'data/ukraine/degurba/UKR_DUG_2024_GRID_L1_R2025A_v1.tif'

with rasterio.open(DEGURBA_TIF) as src:
    raster_crs = src.crs
    print('raster CRS:', raster_crs)

settlements_moll = ukr_admin4.to_crs(raster_crs)
print(f'{len(settlements_moll):,} settlement geometries reprojected to {raster_crs}')

raster CRS: ESRI:54009
29,706 settlement geometries reprojected to ESRI:54009


In [13]:
def _max_degurba(src, geom):
    if geom is None or geom.is_empty:
        return np.nan
    try:
        out_image, _ = rio_mask(src, [geom], crop=True, all_touched=True,
                                 nodata=0, filled=True)
    except ValueError:
        return np.nan  # polygon doesn't overlap the raster extent at all
    valid = out_image[out_image != 0]
    return float(valid.max()) if valid.size else np.nan

with rasterio.open(DEGURBA_TIF) as src:
    ukr_admin4['degurba_l1_ghsl'] = [
        _max_degurba(src, geom) for geom in settlements_moll.geometry
    ]

DEGURBA_LABELS = {1.0: 'Rural', 2.0: 'Urban cluster', 3.0: 'Urban centre'}
ukr_admin4['degurba_l1_ghsl_label'] = ukr_admin4['degurba_l1_ghsl'].map(DEGURBA_LABELS)

print(f'{len(ukr_admin4):,} rows (unchanged)')
print(ukr_admin4['degurba_l1_ghsl'].value_counts(dropna=False))

29,706 rows (unchanged)
degurba_l1_ghsl
1.0    27827
2.0     1698
3.0      181
Name: count, dtype: int64


In [14]:
unmatched = ukr_admin4[ukr_admin4['degurba_l1_ghsl'].isna()]
print(f'{len(unmatched):,} / {len(ukr_admin4):,} settlements have no DEGURBA value')
print(unmatched['adm1_name'].value_counts())

0 / 29,706 settlements have no DEGURBA value
Series([], Name: count, dtype: int64)


In [15]:
# Spot-checks: с. Банилів (rural village, already population-spot-checked earlier) should be
# rural (1); an oblast-capital city row should be urban cluster/centre (2 or 3).
known_pcode = 'UA73020010010031879'[:12]
print(ukr_admin4.loc[ukr_admin4['adm4_pcode'] == known_pcode,
                     ['adm4_name', 'adm4_type', 'degurba_l1_ghsl', 'degurba_l1_ghsl_label']])

city_sample = ukr_admin4[ukr_admin4['adm4_type'] == 'City'].nlargest(5, 'population')
print(city_sample[['adm4_name', 'adm1_name', 'population', 'degurba_l1_ghsl', 'degurba_l1_ghsl_label']])

      adm4_name adm4_type  degurba_l1_ghsl degurba_l1_ghsl_label
27778   Banyliv   Village              1.0                 Rural
      adm4_name        adm1_name  population  degurba_l1_ghsl  \
29704      Kyiv             Kyiv   2952301.0              3.0   
24541   Kharkiv       Kharkivska   1421125.0              3.0   
17064     Odesa           Odeska   1010537.0              3.0   
3639     Dnipro  Dnipropetrovska    968502.0              3.0   
5599    Donetsk         Donetska    901645.0              3.0   

      degurba_l1_ghsl_label  
29704          Urban centre  
24541          Urban centre  
17064          Urban centre  
3639           Urban centre  
5599           Urban centre  


In [16]:
OUT_GPKG = 'data/ukraine/ukr_admin4_population.gpkg'
ukr_admin4.to_file(OUT_GPKG, layer='settlements', driver='GPKG')
print(f'Wrote {OUT_GPKG}, layer=settlements, {len(ukr_admin4):,} rows, '
      f'columns: {list(ukr_admin4.columns)}')

Wrote data/ukraine/ukr_admin4_population.gpkg, layer=settlements, 29,706 rows, columns: ['adm4_name', 'adm4_name1', 'adm4_name2', 'adm4_name3', 'adm4_pcode', 'adm3_name', 'adm3_name1', 'adm3_name2', 'adm3_name3', 'adm3_pcode', 'adm2_name', 'adm2_name1', 'adm2_name2', 'adm2_name3', 'adm2_pcode', 'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang', 'lang1', 'lang2', 'lang3', 'center_lat', 'center_lon', 'adm4_type', 'adm4_type1', 'geometry', 'population', 'degurba_l1_ghsl', 'degurba_l1_ghsl_label']


### Reload the persisted GeoPackage

Fresh read from disk (not the in-memory `ukr_admin4`) to confirm the write round-trips correctly -- row count, columns, and the new population/DEGURBA values all intact.

In [17]:
ukr_admin4_reloaded = gpd.read_file('data/ukraine/ukr_admin4_population.gpkg', layer='settlements')

print(f'{len(ukr_admin4_reloaded):,} rows, {len(ukr_admin4_reloaded.columns)} columns')
print('CRS:', ukr_admin4_reloaded.crs)
print(ukr_admin4_reloaded[['adm4_name', 'adm4_type', 'population', 'degurba_l1_ghsl', 'degurba_l1_ghsl_label']].sample(5, random_state=0))
ukr_admin4_reloaded.head()

29,706 rows, 41 columns
CRS: EPSG:4326
               adm4_name adm4_type  population  degurba_l1_ghsl  \
11627            Yurivka   Village       108.0              1.0   
252          Dvorichchia   Village         NaN              1.0   
28809           Karabany   Village        88.0              1.0   
13284  Novooleksandrivka   Village        77.0              1.0   
9607            Novhorod   Village       170.0              1.0   

      degurba_l1_ghsl_label  
11627                 Rural  
252                   Rural  
28809                 Rural  
13284                 Rural  
9607                  Rural  


,adm4_name,adm4_name1,adm4_name2,adm4_name3,adm4_pcode,adm3_name,adm3_name1,adm3_name2,adm3_name3,adm3_pcode,...,lang2,lang3,center_lat,center_lon,adm4_type,adm4_type1,population,degurba_l1_ghsl,degurba_l1_ghsl_label,geometry
0,Andriivka,Андріївка,None,None,UA0102001001,Sevastopol,Севастополь,None,None,UA0102001,...,None,None,NaN,NaN,Village,Село,NaN,1.0,Rural,"MULTIPOLYGON (((33.56526 44.81991, 33.56445 44..."
1,Soniachnyi,Сонячний,None,None,UA0102001002,Sevastopol,Севастополь,None,None,UA0102001,...,None,None,NaN,NaN,Village,Село,NaN,1.0,Rural,"MULTIPOLYGON (((33.61954 44.78229, 33.6193 44...."
2,Aromatne,Ароматне,None,None,UA0102003001,Aromatnenska,Ароматненська,None,None,UA0102003,...,None,None,NaN,NaN,Village,Село,NaN,1.0,Rural,"MULTIPOLYGON (((33.8678 44.80631, 33.86871 44...."
3,Viktorivka,Вікторівка,None,None,UA0102003002,Aromatnenska,Ароматненська,None,None,UA0102003,...,None,None,NaN,NaN,Village,Село,NaN,1.0,Rural,"MULTIPOLYGON (((33.80358 44.79314, 33.80644 44..."
4,Malovydne,Маловидне,None,None,UA0102003003,Aromatnenska,Ароматненська,None,None,UA0102003,...,None,None,NaN,NaN,Village,Село,NaN,1.0,Rural,"MULTIPOLYGON (((33.81269 44.79064, 33.81182 44..."


## 4. Hromada population — join onto `ukr_admin3`

`ukr_admin3.shp` is the **hromada** (територіальна громада) level -- 1,769 rows, the actual
EU-LAU equivalent, one level coarser than the settlements handled above. Population comes
from the same State Statistics Service table used for `ukr_admin4`, so no new input file is
needed: the hromada subtotals are already sitting in it as the third tier of the per-sheet
hierarchy.

A KATOTTG code is 19 characters and decomposes positionally:

```
UA | 05 | 02 | 001 | 001 | 0058853
   | obl | rai | hrom | sett | serial
     [2:4] [4:6] [6:9] [9:12]  [12:]
```

so `adm3_pcode == code[:9]` here, exactly parallel to `adm4_pcode == code[:12]` above --
again a plain prefix join, no name matching.

Two positional filters isolate the hromada tier:

- `code[9:12] == '000'` -- the settlement segment is empty, i.e. this is *not* a leaf row.
  (This replaces the "усього" name test used for admin4; the code is more reliable than the
  name, and unlike at settlement level the totals are exactly what we want here.)
- `code[6:9] != '000'` -- the hromada segment *is* filled, which drops the raion and oblast
  total rows that also survive the first filter.

**One deliberate exception.** Kyiv and Sevastopol are cities with oblast status: their
oblast, raion and hromada tiers are the same territory, so their only row in the xlsx is
coded at the oblast level (`UA80000000000093317` for `м. Київ`) yet legitimately *is* the
hromada. They are whitelisted by pcode. This exception must stay narrow -- a plain
`code[6:9] == '000'` allowance would also let `UA3200000` **Chornobyl Exclusion Zone**
(a hromada-tier row in the shapefile that has no entry in the xlsx at all) silently swallow
the *Kyiv oblast* total of 1,795,079 instead of being left `NaN`.

In [18]:
# Reuses POP_XLSX / pop_xl / region_sheets from the admin4 section above.
# dtype={'code': str} is explicit here: the positional slicing below depends on the code
# keeping its full 19-character form (same leading-zero hazard flagged for the GDP file).
pop_raw_h = pd.concat(
    [pop_xl.parse(s, header=0, skiprows=[1], usecols=[0, 1, 2],
                  dtype={'code': str}).assign(sheet=s)
     for s in region_sheets],
    ignore_index=True,
)

CITY_OBLAST_PCODES = {'UA8000000', 'UA8500000'}  # Kyiv, Sevastopol -- see note above

_code = pop_raw_h['code']
hromada_pop = pop_raw_h[
    _code.notna()
    & (_code.str[9:12] == '000')                                    # not a settlement leaf
    & ((_code.str[6:9] != '000') | _code.str[:9].isin(CITY_OBLAST_PCODES))
].copy()
hromada_pop['adm3_pcode'] = hromada_pop['code'].str[:9]
hromada_pop['data'] = pd.to_numeric(hromada_pop['data'], errors='coerce')

print(f'{len(hromada_pop):,} hromada rows isolated (expect 1,470)')
assert hromada_pop['adm3_pcode'].duplicated().sum() == 0, 'adm3_pcode is not unique'
hromada_pop[['attributes', 'code', 'adm3_pcode', 'data', 'sheet']].head()

1,470 hromada rows isolated (expect 1,470)


,attributes,code,adm3_pcode,data,sheet
6,"Агрономічна сільська ТГ, усього",UA05020010000053508,UA0502001,9361.0,Вінницька область
15,"Вінницька міська ТГ, усього",UA05020030000031457,UA0502003,387439.0,Вінницька область
27,"Вороновицька селищна ТГ, усього",UA05020050000066991,UA0502005,17127.0,Вінницька область
51,"Гніванська міська ТГ, усього",UA05020070000010139,UA0502007,18446.0,Вінницька область
65,"Іллінецька міська ТГ, усього",UA05020090000067118,UA0502009,21123.0,Вінницька область


In [19]:
ukr_admin3 = gpd.read_file('data/ukraine/admin_boundaries/ukr_admin3.shp')
n_hromadas = len(ukr_admin3)
print(f'{n_hromadas:,} hromada geometries loaded (expect 1,769), CRS = {ukr_admin3.crs}')

ukr_admin3 = ukr_admin3.merge(
    hromada_pop[['adm3_pcode', 'data']].rename(columns={'data': 'population'}),
    on='adm3_pcode', how='left',
)
assert len(ukr_admin3) == n_hromadas, 'left-merge on a unique key changed the row count'
print(f'{len(ukr_admin3):,} rows after merge (unchanged)')

1,769 hromada geometries loaded (expect 1,769), CRS = EPSG:4326
1,769 rows after merge (unchanged)


### Match-rate check

299 hromadas get no population value, and all 299 are explained:

- **297 Crimean + 1 Sevastopol** -- the xlsx has no Crimea/Sevastopol sheet at all, the same
  occupation-exclusion already seen in the GDP file and in the admin4 join above.
- **1 Kyivska** -- `UA3200000` Chornobyl Exclusion Zone, an administrative territory with no
  resident population, which the statistics table simply does not list. `NaN` is the correct
  answer here, not zero and certainly not the oblast total (see the note above).

In [20]:
matched = ukr_admin3['population'].notna().sum()
unmatched = ukr_admin3[ukr_admin3['population'].isna()]
print(f'{matched:,} / {len(ukr_admin3):,} hromadas matched a population value')
print(f'{len(unmatched):,} unmatched -- breakdown by oblast (adm1_name):')
print(unmatched['adm1_name'].value_counts())
print('\nnon-Crimean unmatched:')
print(unmatched.loc[unmatched['adm1_name'] != 'Autonomous Republic of Crimea',
                    ['adm3_pcode', 'adm3_name', 'adm1_name']].to_string(index=False))

# Reverse check: every hromada code in the xlsx must land on a shapefile pcode.
orphan_xlsx = hromada_pop[~hromada_pop['adm3_pcode'].isin(ukr_admin3['adm3_pcode'])]
print(f'\n{len(orphan_xlsx)} xlsx hromada codes have no matching admin3 pcode (expect 0)')
assert matched == 1470 and len(orphan_xlsx) == 0

1,470 / 1,769 hromadas matched a population value
299 unmatched -- breakdown by oblast (adm1_name):
adm1_name
Autonomous Republic of Crimea    297
Kyivska                            1
Sevastopol                         1
Name: count, dtype: int64

non-Crimean unmatched:
adm3_pcode                adm3_name  adm1_name
 UA3200000 Chornobyl Exclusion Zone    Kyivska
 UA8500000               Sevastopol Sevastopol

0 xlsx hromada codes have no matching admin3 pcode (expect 0)


In [21]:
# Spot-check: Kyiv city (UA8000000) is the whitelisted city-with-oblast-status row, and the
# hromada containing с. Банилів (the admin4 spot-check village) is UA7302001 = its code[:9].
print(ukr_admin3.loc[ukr_admin3['adm3_pcode'].isin(['UA8000000', 'UA7302001']),
                     ['adm3_pcode', 'adm3_name', 'adm3_name1', 'population']].to_string(index=False))
assert (ukr_admin3.loc[ukr_admin3['adm3_pcode'] == 'UA8000000', 'population'] == 2952301).all()
print('spot-check OK')

adm3_pcode  adm3_name  adm3_name1  population
 UA7302001 Banylivska Банилівська      7309.0
 UA8000000       Kyiv        Київ   2952301.0
spot-check OK


## 4b. Hromada DEGURBA — urban by containment, town/rural by area share

Same raster as the settlement section above (`UKR_DUG_2024_GRID_L1_R2025A_v1.tif`, 1 km,
`ESRI:54009`, 1=rural / 2=urban cluster / 3=urban centre, 0=nodata), classified in two
independent steps because the two questions are not the same question.

**Urban centre (3) — by containment.** The hromada wholly contains at least one urban-centre
cell; or, for a polygon too small to contain any whole cell, its all-touched maximum is 3.
Deriving "wholly contained" needs a small trick, since `rasterio` has no flag for it: the
cells a polygon covers but does not *cut* are
`all_touched(polygon) & ~all_touched(polygon.boundary)`. Using `geom.boundary` rather than the
exterior ring keeps that correct for the 150 MultiPolygon hromadas and for interior rings.

**Town (2) vs rural (1) — by area share.** A hromada is a town if
`(town_area + urban_area) / covered_area >= DEGURBA_TOWN_AREA_SHARE` (0.15), measured as the
**exact intersection area** of each 1 km cell with the polygon. The geometry is already in
Mollweide (equal-area, metres) for the raster read, so these areas are directly comparable
with no extra projection step.

Three notes on why the rule is shaped this way:

- **A containment maximum is wrong here.** It was the original rule, and it made a single
  contained town cell anywhere in a 339 km²-average hromada enough to call the whole hromada
  a Town — 687 of 1,769 came out Town that way.
- **An area *majority* is also wrong.** Comparing town area against rural area directly
  collapses Town to 30 and moves 657 hromadas (15.9M residents) to Rural, because a hromada
  is a town plus its rural hinterland and the hinterland almost always wins on area. The
  survivors were mostly 5–36 km² Crimean resort hromadas, i.e. the test selected for *small
  polygons* rather than for towns, demoting Lysychansk (111k residents), Kadiivka (215k),
  Brovary (120k), Kalush (87k) and Berdiansk (111k).
- **A share threshold asks the right question** — "is there a town here" rather than "is the
  hinterland bigger" — and keeps all four of those named towns. The share itself is persisted
  as `degurba_nonrural_share` so the threshold can be retuned without recomputing anything:
  0.10 gives 162 towns, 0.15 gives 114, 0.20 gives 83, 0.25 gives 67.

The denominator is *covered* area, not polygon area: the raster has nodata over sea, so
coastal hromadas are only partly covered (as low as 28%) and dividing by the full polygon
area would understate their built-up share.

`degurba_l1_pop` alongside it is an independent second opinion — the official EU DEGURBA LAU
rule, on population rather than area. See its own cell below.

In [22]:
import shapely
from rasterio import features
from rasterio.windows import from_bounds, Window


def _degurba_zonal(src, geom):
    """(max class wholly inside geom, max class geom touches, {class: area m2 inside geom}).

    The two maxima drive the urban-centre test, the per-class areas drive the town/rural
    split; both come out of one windowed read so the raster is only walked once per hromada.
    """
    if geom is None or geom.is_empty:
        return np.nan, np.nan, {}
    # Read a 1-cell-padded window around the polygon rather than the whole raster, so the
    # rasterisations below stay small; boundless covers polygons at the raster edge.
    w = from_bounds(*geom.bounds, transform=src.transform).round_lengths().round_offsets()
    w = Window(w.col_off - 1, w.row_off - 1, w.width + 2, w.height + 2)
    data = src.read(1, window=w, boundless=True, fill_value=0)
    tr = src.window_transform(w)

    touched = features.geometry_mask([geom], data.shape, tr, all_touched=True, invert=True)
    edge = features.geometry_mask([geom.boundary], data.shape, tr, all_touched=True, invert=True)
    inside = touched & ~edge          # covered by the polygon, but not cut by its border

    vi, vt = data[inside], data[touched]
    vi, vt = vi[vi != 0], vt[vt != 0]   # 0 is nodata

    # Exact per-class area: rebuild the touched cells as boxes and intersect them with the
    # polygon. Vectorised over the whole window at once -- shapely 2's box/intersection/area
    # all broadcast over arrays, so this is one call per hromada, not one per cell.
    sel = touched & (data != 0)
    rows, cols = np.nonzero(sel)
    by_class = {}
    if rows.size:
        x0 = tr.c + cols * tr.a
        y0 = tr.f + rows * tr.e         # tr.e is negative, so y0 is each cell's TOP edge
        cells = shapely.box(x0, y0 + tr.e, x0 + tr.a, y0)
        areas = shapely.area(shapely.intersection(cells, geom))
        cls = data[sel]
        by_class = {int(k): float(areas[cls == k].sum()) for k in np.unique(cls)}

    return (float(vi.max()) if vi.size else np.nan,
            float(vt.max()) if vt.size else np.nan,
            by_class)


# ~2 minutes for all 1,769 hromadas -- the per-cell intersection is the expensive part.
with rasterio.open(DEGURBA_TIF) as src:
    hromadas_moll = ukr_admin3.to_crs(src.crs)
    print(f'{len(hromadas_moll):,} hromada geometries reprojected to {src.crs}')
    _zonal = [_degurba_zonal(src, geom) for geom in hromadas_moll.geometry]

contained = pd.Series([z[0] for z in _zonal], index=ukr_admin3.index)
touched = pd.Series([z[1] for z in _zonal], index=ukr_admin3.index)
_class_area = (pd.DataFrame([z[2] for z in _zonal], index=ukr_admin3.index)
               .reindex(columns=[1, 2, 3]).fillna(0.0))
_covered = _class_area.sum(axis=1)

# Covered area can fall short of the polygon (nodata over sea) but must never exceed it.
_ratio = _covered / hromadas_moll.area.values
print(f'covered/polygon area -- median {_ratio.median():.4f}, min {_ratio.min():.4f} '
      f'(coastal hromadas, sea is nodata), max {_ratio.max():.6f}')
assert _ratio.max() <= 1 + 1e-9, 'covered area exceeds polygon area'
print('\nwholly-contained-cell maximum:')
print(contained.value_counts(dropna=False).sort_index())

1,769 hromada geometries reprojected to ESRI:54009
covered/polygon area -- median 1.0000, min 0.2840 (coastal hromadas, sea is nodata), max 1.000000

wholly-contained-cell maximum:
1.0    1003
2.0     683
3.0      73
NaN      10
Name: count, dtype: int64


In [23]:
DEGURBA_TOWN_AREA_SHARE = 0.15

# Urban centre: a wholly-contained urban-centre cell, or -- for the 10 hromadas too small to
# contain any whole cell -- an all-touched maximum of 3. This is the containment test only;
# the area share below never promotes a hromada to urban centre.
_urban = (contained == 3) | (contained.isna() & (touched == 3))

ukr_admin3['degurba_rural_km2'] = _class_area[1] / 1e6
ukr_admin3['degurba_town_km2'] = _class_area[2] / 1e6
ukr_admin3['degurba_urban_km2'] = _class_area[3] / 1e6
ukr_admin3['degurba_nonrural_share'] = ((_class_area[2] + _class_area[3]) / _covered).fillna(0.0)

ukr_admin3['degurba_l1_ghsl'] = np.where(
    _urban, 3.0,
    np.where(ukr_admin3['degurba_nonrural_share'] >= DEGURBA_TOWN_AREA_SHARE, 2.0, 1.0))
ukr_admin3['degurba_l1_ghsl_fallback'] = contained.isna() & touched.notna()
ukr_admin3['degurba_l1_ghsl_label'] = ukr_admin3['degurba_l1_ghsl'].map(DEGURBA_LABELS)

print(f'{len(ukr_admin3):,} rows (unchanged)')
print(ukr_admin3['degurba_l1_ghsl'].value_counts().sort_index())
print(f"\n{int(ukr_admin3['degurba_l1_ghsl_fallback'].sum())} hromadas had no wholly-contained "
      f"cell, so their urban test used the all-touched maximum")
assert ukr_admin3['degurba_l1_ghsl'].notna().all(), 'some hromada is unclassified'
assert int(_urban.sum()) == 74, 'the urban-centre set must not move -- only town/rural changed'
print(f"\nnon-rural area share -- median {ukr_admin3['degurba_nonrural_share'].median():.4f}, "
      f"threshold {DEGURBA_TOWN_AREA_SHARE}")

1,769 rows (unchanged)
degurba_l1_ghsl
1.0    1581
2.0     114
3.0      74
Name: count, dtype: int64

10 hromadas had no wholly-contained cell, so their urban test used the all-touched maximum

non-rural area share -- median 0.0013, threshold 0.15


### Alternative: `degurba_l1_pop` — the official DEGURBA LAU rule, on population

The area rule above is a proxy: it asks how much of the hromada is built up. The EU's actual
LAU-level DEGURBA definition asks where the *people* are — a LAU is a city if more than half
its residents live in urban centres, a town/suburb if more than half live in urban clusters or
centres, and rural otherwise.

That is computable here without new inputs, because the settlement section earlier in this
notebook already produced both a population and a DEGURBA class per settlement in
`ukr_admin4_population.gpkg`. Rolling those up by `adm4_pcode[:9] == adm3_pcode` gives
population by class per hromada, and the thresholds apply directly.

It is kept as a **separate column rather than merged into `degurba_l1_ghsl`**, for two
reasons. It is only defined for 1,470 of the 1,769 hromadas — the 297 Crimean, Sevastopol and
Chornobyl Exclusion Zone rows have no settlement population data at all, and backfilling them
from the area rule would destroy the whole point of having a second, independent opinion. And
the two rules genuinely disagree: they agree on 1,163 of the 1,470 comparable hromadas (79%),
with the population rule calling ~290 more of them towns, so which one is right depends on
what a downstream consumer is measuring.

In [24]:
UKR_ADMIN4_GPKG = 'data/ukraine/ukr_admin4_population.gpkg'

_s4 = gpd.read_file(UKR_ADMIN4_GPKG, layer='settlements', ignore_geometry=True)
_s4['adm3_pcode'] = _s4['adm4_pcode'].str[:9]
_s4['_pop'] = pd.to_numeric(_s4['population'], errors='coerce').fillna(0.0)

_pop_by_class = (_s4.pivot_table(index='adm3_pcode', columns='degurba_l1_ghsl',
                                 values='_pop', aggfunc='sum')
                 .reindex(columns=[1.0, 2.0, 3.0]).fillna(0.0))
_pop_tot = _pop_by_class.sum(axis=1)

# Official DEGURBA LAU thresholds, most-urban test last so it wins.
_pop_cls = pd.Series(1.0, index=_pop_by_class.index)
_pop_cls[(_pop_by_class[2.0] + _pop_by_class[3.0]) / _pop_tot > 0.5] = 2.0
_pop_cls[_pop_by_class[3.0] / _pop_tot > 0.5] = 3.0
_pop_cls[_pop_tot == 0] = np.nan     # no settlement population data at all -> not guessed

ukr_admin3['degurba_l1_pop'] = ukr_admin3['adm3_pcode'].map(_pop_cls)
ukr_admin3['degurba_l1_pop_label'] = ukr_admin3['degurba_l1_pop'].map(DEGURBA_LABELS)

print(ukr_admin3['degurba_l1_pop'].value_counts(dropna=False).sort_index())
_unclassified = ukr_admin3[ukr_admin3['degurba_l1_pop'].isna()]
print(f'\n{len(_unclassified)} hromadas unclassified (no settlement population):')
print(_unclassified['adm1_name'].value_counts().to_string())
assert set(_unclassified.loc[
    ~_unclassified['adm1_name'].isin(['Autonomous Republic of Crimea', 'Sevastopol']),
    'adm3_pcode']) == {'UA3200000'}, 'unexpected hromada has no population data'

_both = ukr_admin3['degurba_l1_pop'].notna()
_agree = int((ukr_admin3.loc[_both, 'degurba_l1_ghsl']
              == ukr_admin3.loc[_both, 'degurba_l1_pop']).sum())
print()
print(f'the two rules agree on {_agree:,} of {int(_both.sum()):,} comparable hromadas')

degurba_l1_pop
1.0    1033
2.0     372
3.0      65
NaN     299
Name: count, dtype: int64

299 hromadas unclassified (no settlement population):
adm1_name
Autonomous Republic of Crimea    297
Kyivska                            1
Sevastopol                         1

the two rules agree on 1,163 of 1,470 comparable hromadas


In [25]:
# Kyiv/Sevastopol are urban centres; Kotsiubynska is the small-polygon fallback case.
# Lysychanska/Brovarska/Kaluska/Berdianska are the towns an area *majority* rule would have
# wrongly demoted -- they clear the share threshold. Kadiivska is the closest miss at 0.127
# despite having the most town fabric of any hromada in absolute terms (99 km2), because it
# is also one of the largest (785 km2) -- the case to look at first if the threshold moves.
_named = ['Kyiv', 'Sevastopol', 'Kotsiubynska', 'Lysychanska', 'Brovarska', 'Kaluska',
          'Berdianska', 'Kadiivska']
print(ukr_admin3[ukr_admin3['adm3_name'].isin(_named)][
    ['adm3_name', 'area_sqkm', 'degurba_town_km2', 'degurba_nonrural_share',
     'degurba_l1_ghsl_label', 'degurba_l1_pop_label']]
      .sort_values('degurba_nonrural_share', ascending=False).to_string(index=False))

for _nm in ['Lysychanska', 'Brovarska', 'Kaluska', 'Berdianska']:
    assert (ukr_admin3.loc[ukr_admin3['adm3_name'] == _nm, 'degurba_nonrural_share'] >= 0.15).all(), _nm
assert (ukr_admin3.loc[ukr_admin3['adm3_name'] == 'Kadiivska', 'degurba_nonrural_share'] < 0.15).all()

print('\n5 hromadas closest to the threshold from each side (retuning candidates):')
_near = ukr_admin3.loc[~_urban, ['adm3_name', 'adm1_name', 'area_sqkm',
                                 'degurba_nonrural_share', 'degurba_l1_ghsl_label']]
_near = _near.assign(_d=(_near['degurba_nonrural_share'] - DEGURBA_TOWN_AREA_SHARE))
print(_near.reindex(_near['_d'].abs().sort_values().index).head(10)
      .drop(columns='_d').to_string(index=False))

print('\narea-share rule (rows) x population-share rule (cols):')
print(pd.crosstab(ukr_admin3['degurba_l1_ghsl_label'], ukr_admin3['degurba_l1_pop_label'],
                  dropna=False).to_string())

   adm3_name  area_sqkm  degurba_town_km2  degurba_nonrural_share degurba_l1_ghsl_label degurba_l1_pop_label
Kotsiubynska   2.516054          1.782424                0.999343          Urban centre         Urban centre
        Kyiv 823.190953         93.512450                0.626272          Urban centre         Urban centre
   Brovarska 125.017558         46.927116                0.375840         Urban cluster        Urban cluster
  Sevastopol 848.850630        125.864631                0.245948          Urban centre                  NaN
     Kaluska 266.061171         46.524841                0.175585         Urban cluster        Urban cluster
 Lysychanska 409.431549         68.550855                0.167579         Urban cluster        Urban cluster
  Berdianska 250.231752         41.363463                0.165374         Urban cluster        Urban cluster
   Kadiivska 785.092655         99.481295                0.126818                 Rural        Urban cluster

5 hromadas closest

In [26]:
OUT_GPKG_3 = 'data/ukraine/ukr_admin3_population.gpkg'
ukr_admin3.to_file(OUT_GPKG_3, layer='hromadas', driver='GPKG')
print(f'Wrote {OUT_GPKG_3}, layer=hromadas, {len(ukr_admin3):,} rows, '
      f'columns: {list(ukr_admin3.columns)}')

Wrote data/ukraine/ukr_admin3_population.gpkg, layer=hromadas, 1,769 rows, columns: ['adm3_name', 'adm3_name1', 'adm3_name2', 'adm3_name3', 'adm3_pcode', 'adm2_name', 'adm2_name1', 'adm2_name2', 'adm2_name3', 'adm2_pcode', 'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang', 'lang1', 'lang2', 'lang3', 'center_lat', 'center_lon', 'adm3_name_', 'adm3_nam_1', 'geometry', 'population', 'degurba_rural_km2', 'degurba_town_km2', 'degurba_urban_km2', 'degurba_nonrural_share', 'degurba_l1_ghsl', 'degurba_l1_ghsl_fallback', 'degurba_l1_ghsl_label', 'degurba_l1_pop', 'degurba_l1_pop_label']


### Reload the persisted GeoPackage

Same round-trip check as for the settlement layer -- fresh read from disk rather than the
in-memory `ukr_admin3`, confirming row count, CRS and the new columns all survive the write.

In [27]:
ukr_admin3_reloaded = gpd.read_file('data/ukraine/ukr_admin3_population.gpkg', layer='hromadas')

print(f'{len(ukr_admin3_reloaded):,} rows, {len(ukr_admin3_reloaded.columns)} columns')
print('CRS:', ukr_admin3_reloaded.crs)
assert len(ukr_admin3_reloaded) == len(ukr_admin3)
print(ukr_admin3_reloaded[['adm3_name', 'adm1_name', 'population', 'degurba_l1_ghsl',
                           'degurba_l1_ghsl_label']].sample(5, random_state=0))
ukr_admin3_reloaded.head()

1,769 rows, 43 columns
CRS: EPSG:4326
               adm3_name         adm1_name  population  degurba_l1_ghsl  \
1168           Tairovska            Odeska     14719.0              3.0   
670   Velykobereznianska       Zakarpatska      9969.0              1.0   
412           Torchynska          Volynska     12034.0              1.0   
1056      Vilnozaporizka       Mykolaivska      5420.0              1.0   
789             Vyhodska  Ivano-Frankivska     18028.0              1.0   

     degurba_l1_ghsl_label  
1168          Urban centre  
670                  Rural  
412                  Rural  
1056                 Rural  
789                  Rural  


,adm3_name,adm3_name1,adm3_name2,adm3_name3,adm3_pcode,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,...,degurba_rural_km2,degurba_town_km2,degurba_urban_km2,degurba_nonrural_share,degurba_l1_ghsl,degurba_l1_ghsl_fallback,degurba_l1_ghsl_label,degurba_l1_pop,degurba_l1_pop_label,geometry
0,Aromatnenska,Ароматненська,None,None,UA0102003,Bakhchysaraiskyi,Бахчисарайський,None,None,UA0102,...,87.731680,2.225769,0.0,0.024742,1.0,False,Rural,NaN,NaN,"MULTIPOLYGON (((33.82716 44.85789, 33.83145 44..."
1,Bakhchysaraiska,Бахчисарайська,None,None,UA0102005,Bakhchysaraiskyi,Бахчисарайський,None,None,UA0102,...,37.391491,17.996184,0.0,0.324913,2.0,False,Urban cluster,NaN,NaN,"MULTIPOLYGON (((33.87243 44.7769, 33.86937 44...."
2,Verkhorichenska,Верхоріченська,None,None,UA0102009,Bakhchysaraiskyi,Бахчисарайський,None,None,UA0102,...,357.738589,0.715426,0.0,0.001996,1.0,False,Rural,NaN,NaN,"MULTIPOLYGON (((34.23224 44.79825, 34.24962 44..."
3,Vilinska,Вілінська,None,None,UA0102011,Bakhchysaraiskyi,Бахчисарайський,None,None,UA0102,...,51.029541,0.000000,0.0,0.000000,1.0,False,Rural,NaN,NaN,"MULTIPOLYGON (((33.69214 44.87577, 33.69644 44..."
4,Holubynska,Голубинська,None,None,UA0102013,Bakhchysaraiskyi,Бахчисарайський,None,None,UA0102,...,212.104923,0.000000,0.0,0.000000,1.0,False,Rural,NaN,NaN,"MULTIPOLYGON (((33.9474 44.61931, 33.9477 44.6..."


## 5. Outputs check

What `00_extract_base_data_eu` relies on: both GeoPackages present, every hromada classified,
population missing only where the source has none (occupied Crimea and Sevastopol, plus the
Chornobyl Exclusion Zone hromada in Kyivska oblast).

In [28]:
_s4 = gpd.read_file("data/ukraine/ukr_admin4_population.gpkg", layer="settlements", ignore_geometry=True)
_h3 = gpd.read_file("data/ukraine/ukr_admin3_population.gpkg", layer="hromadas", ignore_geometry=True)
print(f"settlements: {len(_s4):,}   hromadas: {len(_h3):,}")
assert len(_h3) == 1769 and _h3["adm3_pcode"].is_unique
assert _h3["degurba_l1_ghsl"].notna().all(), "hromada without degurba_l1_ghsl"
_no_pop = _h3.loc[_h3["population"].isna(), "adm1_name"].value_counts()
print("hromadas without population, by oblast:")
print(_no_pop.to_string())
assert set(_no_pop.index) <= {"Autonomous Republic of Crimea", "Sevastopol", "Kyivska"}
print("degurba_l1_ghsl:", _h3["degurba_l1_ghsl"].value_counts().sort_index().to_dict())

settlements: 29,706   hromadas: 1,769
hromadas without population, by oblast:
adm1_name
Autonomous Republic of Crimea    297
Kyivska                            1
Sevastopol                         1
degurba_l1_ghsl: {1.0: 1581, 2.0: 114, 3.0: 74}
